In [18]:
from pyspark.sql.functions import col, sum as spark_sum

# 1. Tambahkan kolom pendapatan terlebih dahulu
df_transaksi = df_transaksi.withColumn(
    "pendapatan", col("unit_terjual") * col("harga_satuan")
)

# 2. Ringkas total pendapatan per kota
df_ringkasan = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# 3. Join dengan df_target dan hitung persentase pencapaian
df_bagian_a = df_ringkasan.join(df_target, on="kota", how="inner").withColumn(
    "pencapaian_persen",
    (col("total_pendapatan") / col("target_bulanan")) * 100,
)

# 4. Urutkan dari pencapaian tertinggi
df_bagian_a = df_bagian_a.orderBy(col("pencapaian_persen").desc())
df_bagian_a.show()

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



In [17]:
import pandas as pd
from pyspark.sql.functions import col, sum as spark_sum

# 0. Buat df_target dari dictionary target cabang
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}
df_target = spark.createDataFrame(pd.DataFrame(data_target_cabang))

# 1. Tambahkan kolom pendapatan
df_transaksi = df_transaksi.withColumn(
    "pendapatan", col("unit_terjual") * col("harga_satuan")
)

# 2. Ringkas total pendapatan per kota
df_ringkasan = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# 3. Join dengan df_target dan hitung persentase pencapaian
df_bagian_a = df_ringkasan.join(df_target, on="kota", how="inner").withColumn(
    "pencapaian_persen",
    (col("total_pendapatan") / col("target_bulanan")) * 100,
)

# 4. Urutkan dari pencapaian tertinggi
df_bagian_a = df_bagian_a.orderBy(col("pencapaian_persen").desc())
df_bagian_a.show()

[Stage 5:>                                                          (0 + 8) / 8]

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



In [5]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, rank, row_number, when
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Pertemuan5-JoinWindowSQL") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession siap. Versi Spark:", spark.version)

SparkSession siap. Versi Spark: 3.5.9


In [12]:
import numpy as np
import pandas as pd

# 1. Buat dataset transaksi lokal
np.random.seed(55)
n = 500
kategori_list = [
    "Elektronik",
    "Fashion",
    "Makanan & Minuman",
    "Kesehatan & Kecantikan",
    "Rumah Tangga",
]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice(
        [25000, 50000, 75000, 100000, 150000], size=n
    ),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

# 2. Upload ke HDFS (memanggil biner hdfs secara langsung menggunakan absolute path)
!/home/albani/hadoop/bin/hdfs dfs -mkdir -p /user/albani/tugas5
!/home/albani/hadoop/bin/hdfs dfs -put -f transaksi_tugas5.csv /user/albani/tugas5/

# 3. Cek keberadaan file di HDFS
!/home/albani/hadoop/bin/hdfs dfs -ls /user/albani/tugas5/

Found 1 items
-rw-r--r--   1 albani supergroup      19997 2026-09-28 11:12 /user/albani/tugas5/transaksi_tugas5.csv


In [9]:
import numpy as np
import pandas as pd

# Tabel 1: Target & PIC per cabang kota (tabel referensi, dibuat langsung sebagai DataFrame)
data_target_cabang = {
    "kota": ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"],
    "target_bulanan": [45000000, 60000000, 55000000, 40000000, 30000000],
    "pic_cabang": ["Rani", "Joko", "Sari", "Bayu", "Fitri"],
}

# Tabel 2: Data transaksi (disimpan sebagai CSV, lalu diunggah ke HDFS)
np.random.seed(55)
n = 500
kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
data_transaksi_t5 = {
    "order_id": [f"TRX-{i}" for i in range(n)],
    "kategori": np.random.choice(kategori_list, size=n),
    "kota": np.random.choice(kota_list, size=n),
    "unit_terjual": np.random.randint(1, 10, size=n),
    "harga_satuan": np.random.choice([25000, 50000, 75000, 100000, 150000], size=n),
}
pd.DataFrame(data_transaksi_t5).to_csv("transaksi_tugas5.csv", index=False)

!hdfs dfs -mkdir -p /user/albani/tugas5
!hdfs dfs -put -f transaksi_tugas5.csv /user/albani/tugas5/
print("Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/albani/tugas5/transaksi_tugas5.csv")
print("Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.")

/bin/bash: line 1: hdfs: command not found
/bin/bash: line 1: hdfs: command not found
Dataset siap. Tabel transaksi sudah diunggah ke HDFS: /user/albani/tugas5/transaksi_tugas5.csv
Simpan juga 'data_target_cabang' di atas — kalian akan membuatnya menjadi DataFrame sendiri di notebook tugas.


In [13]:
# 2. Baca dataset transaksi dari HDFS (sesuaikan username ke albani)
df_transaksi = spark.read.csv(
    "hdfs://localhost:9000/user/albani/tugas5/transaksi_tugas5.csv",
    header=True,
    inferSchema=True,
)

In [19]:
# Meringkas total pendapatan per kota
df_ringkasan = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# Join dengan df_target dan menghitung pencapaian_persen
df_bagian_a = df_ringkasan.join(df_target, on="kota", how="inner").withColumn(
    "pencapaian_persen",
    (col("total_pendapatan") / col("target_bulanan")) * 100,
)

# Urutkan dari pencapaian tertinggi
df_bagian_a = df_bagian_a.orderBy(col("pencapaian_persen").desc())
df_bagian_a.show()

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



In [20]:
# Meringkas total pendapatan per kota
df_ringkasan = df_transaksi.groupBy("kota").agg(
    spark_sum("pendapatan").alias("total_pendapatan")
)

# Join dengan df_target dan hitung pencapaian_persen
df_bagian_a = df_ringkasan.join(df_target, on="kota", how="inner").withColumn(
    "pencapaian_persen",
    (col("total_pendapatan") / col("target_bulanan")) * 100,
)

# Urutkan dari pencapaian tertinggi
df_bagian_a = df_bagian_a.orderBy(col("pencapaian_persen").desc())
df_bagian_a.show()

+----------+----------------+--------------+----------+------------------+
|      kota|total_pendapatan|target_bulanan|pic_cabang| pencapaian_persen|
+----------+----------------+--------------+----------+------------------+
| Purworejo|        45650000|      30000000|     Fitri|152.16666666666669|
|      Solo|        33475000|      40000000|      Bayu|           83.6875|
|Yogyakarta|        47275000|      60000000|      Joko| 78.79166666666667|
|  Magelang|        31650000|      45000000|      Rani| 70.33333333333334|
|  Semarang|        38175000|      55000000|      Sari|  69.4090909090909|
+----------+----------------+--------------+----------+------------------+



In [21]:
# Agregasi pendapatan per kota dan kategori
df_kota_kategori = df_transaksi.groupBy("kota", "kategori").agg(
    spark_sum("pendapatan").alias("total_pendapatan_kategori")
)

# Window spec per kota diurutkan dari pendapatan tertinggi
window_spec = Window.partitionBy("kota").orderBy(
    col("total_pendapatan_kategori").desc()
)

# Ambil peringkat 1 menggunakan row_number()
df_bagian_b = (
    df_kota_kategori.withColumn("rank", row_number().over(window_spec))
    .filter(col("rank") == 1)
    .drop("rank")
)

df_bagian_b.orderBy("kota").show()

+----------+--------------------+-------------------------+
|      kota|            kategori|total_pendapatan_kategori|
+----------+--------------------+-------------------------+
|  Magelang|Kesehatan & Kecan...|                  7275000|
| Purworejo|Kesehatan & Kecan...|                 10075000|
|  Semarang|        Rumah Tangga|                 11125000|
|      Solo|Kesehatan & Kecan...|                  8425000|
|Yogyakarta|             Fashion|                 13325000|
+----------+--------------------+-------------------------+



In [22]:
# Daftarkan temporary view
df_transaksi.createOrReplaceTempView("transaksi")
df_target.createOrReplaceTempView("target")

# Kueri SQL untuk menghitung jumlah transaksi per kota & PIC
df_bagian_c = spark.sql("""
    SELECT 
        t.kota, 
        tg.pic_cabang, 
        COUNT(t.order_id) AS jumlah_transaksi
    FROM transaksi t
    JOIN target tg ON t.kota = tg.kota
    GROUP BY t.kota, tg.pic_cabang
    ORDER BY jumlah_transaksi DESC
""")

df_bagian_c.show()

+----------+----------+----------------+
|      kota|pic_cabang|jumlah_transaksi|
+----------+----------+----------------+
| Purworejo|     Fitri|             116|
|Yogyakarta|      Joko|             110|
|      Solo|      Bayu|              95|
|  Semarang|      Sari|              93|
|  Magelang|      Rani|              86|
+----------+----------+----------------+

